In [1]:
from PyPDF2 import PdfReader

In [2]:
#Document loader
pdf_reader = PdfReader("data/documents/Retrieval-augmented_generation.pdf")

In [3]:
text = ""

for page in pdf_reader.pages:
    text += page.extract_text()


In [4]:
text

'Retrieval-augmented generation\nRetrieval-augmented generation  (RAG ) is a technique that enables large language models  (LLMs) to\nretrieve and incorporate new information from external data sources.[1][2] With RAG, LLMs first refer to\na specified set of documents, then respond to user queries. These documents supplement information\nfrom the LLM\'s pre-existing training data.[3] This allows LLMs to use domain-specific and/or updated\ninformation that is not available in the training data.[3] For example, this enables LLM-based chatbots  to\naccess internal company data or generate responses based on authoritative sources. The technique was\nfirst proposed in 2020 and has since become a widely adopted approach in modern AI systems.\nRAG improves LLMs by incorporating information retrieval  before generating responses.[4] Unlike\nLLMs that rely on static training data, RAG pulls relevant text from databases, uploaded documents, or\nweb sources.[1] According to Ars Technica , "RAG is

In [5]:
len(text)

17944

In [6]:
cut = text.find('1. "What is retrieval-augmented generation?"')
clean_text = text[:cut] if cut != -1 else text
print(len(text), "->", len(clean_text))

17944 -> 11122


In [7]:
from langchain_text_splitters import RecursiveCharacterTextSplitter

splitter = RecursiveCharacterTextSplitter(chunk_size=800, chunk_overlap=150)
chunks = splitter.split_text(clean_text)
print("Chunks:", len(chunks))

Chunks: 17


In [8]:
import os
from dotenv import load_dotenv

load_dotenv(r"C:\Users\Admin\Desktop\streamlit\RAG-Chatbot\.env", override=True)
os.environ.pop("GEMINI_API_KEY", None)  # பழைய பெயர் இருந்தா நீக்கும்

print("Key found:", bool(os.environ.get("GOOGLE_API_KEY")))

Key found: True


In [9]:
from langchain_google_genai import ChatGoogleGenerativeAI, GoogleGenerativeAIEmbeddings

key = os.environ.get("GOOGLE_API_KEY")
llm = ChatGoogleGenerativeAI(model="gemini-3.8-flash", api_key=key, max_retries=2)
embed = GoogleGenerativeAIEmbeddings(model="models/gemini-embedding-001", api_key=key)
print("llm and embed created")

llm and embed created


In [10]:
from pathlib import Path
from langchain_community.vectorstores import FAISS

INDEX_DIR = "vectorstore/faiss_index"

if Path(INDEX_DIR).exists():
    db = FAISS.load_local(INDEX_DIR, embed, allow_dangerous_deserialization=True)
    print("Loaded saved index")
else:
    db = FAISS.from_texts(chunks, embedding=embed)
    db.save_local(INDEX_DIR)
    print("Built and saved new index")

print("Chunks in index:", len(db.docstore._dict))

C:\Users\Admin\AppData\Local\Temp\ipykernel_10792\4293509880.py:2: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.vectorstores import FAISS


Loaded saved index
Chunks in index: 17


In [11]:
question = "Why do we need RAG?"
for r in db.similarity_search(question, k=3):
    print("-", r.page_content[:120].replace("\n", " "))

- first proposed in 2020 and has since become a widely adopted approach in modern AI systems. RAG improves LLMs by incorpo
- that don't exist, or recommend nonexistent legal cases to lawyers that are looking for citations to support their ar gum
- One limitation is that while RAG reduces the need for frequent model retraining, it does not remove it entirely . Additi


In [15]:
def get_text(response):
    content = response.content
    if isinstance(content, str):
        return content
    return "".join(
        part.get("text", "") for part in content if isinstance(part, dict)
    )

print("get_text ready")

get_text ready


In [12]:
def ask(question):
    results = db.similarity_search(question, k=3)
    context = "\n\n".join(r.page_content for r in results)

    prompt = f"""Answer the question using only the context below.
If the answer is not in the context, say "I don't know".

Context:
{context}

Question: {question}"""

    return get_text(llm.invoke(prompt))

print("ask ready")

ask ready


In [13]:
from typing import TypedDict, List
from langgraph.graph import StateGraph, END
from langchain_core.prompts import ChatPromptTemplate


class RAGState(TypedDict):
    question: str
    docs: List
    answer: str
    grounded: bool
    retries: int


answer_prompt = ChatPromptTemplate.from_template(
    """Answer the question using only the context below.
If the answer is not in the context, say "I don't know".

Context:
{context}

Question: {question}"""
)

grade_prompt = ChatPromptTemplate.from_template(
    """Context:
{context}

Answer:
{answer}

Is the answer fully supported by the context? Reply with only "yes" or "no"."""
)


def retrieve(state: RAGState):
    k = 3 + 2 * state["retries"]
    docs = db.similarity_search(state["question"], k=k)
    return {"docs": docs}


def generate(state: RAGState):
    context = "\n\n".join(d.page_content for d in state["docs"])
    msg = answer_prompt.invoke({"context": context, "question": state["question"]})
    return {"answer": get_text(llm.invoke(msg))}


def grade(state: RAGState):
    context = "\n\n".join(d.page_content for d in state["docs"])
    msg = grade_prompt.invoke({"context": context, "answer": state["answer"]})
    verdict = get_text(llm.invoke(msg)).strip().lower()
    return {"grounded": verdict.startswith("yes"), "retries": state["retries"] + 1}


def decide(state: RAGState):
    if state["grounded"] or state["retries"] >= 2:
        return "end"
    return "retry"


graph = StateGraph(RAGState)
graph.add_node("retrieve", retrieve)
graph.add_node("generate", generate)
graph.add_node("grade", grade)

graph.set_entry_point("retrieve")
graph.add_edge("retrieve", "generate")
graph.add_edge("generate", "grade")
graph.add_conditional_edges("grade", decide, {"retry": "retrieve", "end": END})

rag_app = graph.compile()
print("rag_app ready")

rag_app ready


In [ ]:
print(ask("What is RAG?"))

In [ ]:
result = rag_app.invoke({
     "question": "Why do we need RAG?",
     "docs": [], "answer": "", "grounded": False, "retries": 0
 })
print(result["answer"])
print("Grounded:", result["grounded"], "| Retries:", result["retries"])

In [ ]:
result = rag_app.invoke({
     "question": "Who won the FIFA World Cup in 2022?",
     "docs": [], "answer": "", "grounded": False, "retries": 0
 })
print(result["answer"])
print("Grounded:", result["grounded"], "| Retries:", result["retries"])